# Salary Prediction — Full Training Run (VM)

Runs the complete Phase 4/5 pipeline against the real dataset, with Spark **explicitly configured and verified** for this VM (16 cores, wider tuning grids, concurrent hyperparameter tuning) — see `PLAN.md` §23 #18 for the full reasoning.

**Run cells top to bottom.** The Spark-session cell below prints whether the configured settings actually took effect — read its output before continuing to the training cell.

## 1. Path and working directory

In [ ]:
import sys, os

PROJECT_ROOT = "/home/linuxu/project"  # adjust if this VM's checkout lives elsewhere

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

## 1a. Install missing Python packages (one-time)

If a later cell fails with `ModuleNotFoundError`, add `%pip install <package-name>` here and re-run from the top after restarting the kernel.

In [ ]:
%pip install python-dotenv

### If you hit a numpy/scipy import error (`_ARRAY_API not found`, `AttributeError` from `scipy`, a `UserWarning` about needing `numpy<1.23`, etc.)

PySpark ML's linear algebra internals (`pyspark.ml.linalg`) use `scipy` for sparse vector operations. This VM's conda environment has an older `scipy` built against numpy 1.x, so if numpy has been upgraded to 2.x here (e.g. by another notebook's environment fix), `scipy` breaks with a binary-compatibility error. Run the cell below once, then **restart the kernel** (a live kernel keeps old binaries loaded in memory even after new ones are installed on disk) and re-run this notebook from the top.

In [ ]:
# Only run this if a later cell fails with a numpy/scipy binary-compatibility
# error. Then: Kernel -> Restart Kernel, and re-run from the top - a restart is
# required, installing alone does not fix an already-running kernel's in-memory
# state.
%pip install --upgrade --force-reinstall numpy scipy

## 2. One-time setup — `.env` and the raw CSV

Idempotent: skips anything already done on a previous run.

In [ ]:
import subprocess

if not os.path.exists(".env"):
    subprocess.run(["cp", ".env.example", ".env"])
    print("Created .env from .env.example.")
else:
    print(".env already exists (not overwritten).")

print()
print(open(".env").read())

**Check the `.env` contents printed above.** For this run you want:
- `SPARK_DRIVER_MEMORY=6g` (or higher, if the VM has RAM to spare)
- `SPARK_SHUFFLE_PARTITIONS=16` (match the VM's actual core count — run `nproc` in a terminal to confirm it's really 16)
- `TUNING_PARALLELISM=4` (concurrent hyperparameter-search fits; drop to `1` only if this run hits a `ConnectionRefusedError` — see `PLAN.md` §23 #13/#18)

If `.env` already existed from a previous session and is missing these or has different values, edit it now (in a terminal, or via JupyterLab's file browser), then re-run the cell above to confirm.

In [ ]:
if not os.path.exists("data/raw/survey_results_public.csv"):
    result = subprocess.run(
        ["unzip", "-o", "data.zip", "survey_results_public.csv", "-d", "data/raw/"],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
else:
    print("data/raw/survey_results_public.csv already exists.")

## 3. Spark session — force a fresh one so every setting actually applies

A Spark session created earlier in this kernel (or auto-attached by the notebook environment) has its driver memory and other JVM-level settings fixed **at creation time** — they cannot be changed on an already-running session, no matter what `config/settings.py` says. This is exactly what caused a training run to silently use the wrong memory settings before.

The cell below stops any existing session, builds a fresh one from this project's config, and then **prints the actual values Spark is using** so you can confirm before spending time on a full training run.

In [ ]:
from pyspark.sql import SparkSession

try:
    SparkSession.builder.getOrCreate().stop()
    print("Stopped an existing Spark session.")
except Exception as exc:
    print("No existing session to stop (or stop failed) - continuing:", exc)

In [ ]:
from src.common.spark_session import get_spark_session
from config import settings

spark = get_spark_session(app_name="SalaryModelTraining")

actual = {
    "spark.driver.memory": spark.sparkContext.getConf().get("spark.driver.memory"),
    "spark.sql.shuffle.partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    "spark.default.parallelism (actual cluster parallelism)": spark.sparkContext.defaultParallelism,
    "spark.master": spark.sparkContext.master,
}

print("Expected (from config/settings.py, i.e. your .env):")
print("  SPARK_DRIVER_MEMORY      =", settings.SPARK_DRIVER_MEMORY)
print("  SPARK_SHUFFLE_PARTITIONS =", settings.SPARK_SHUFFLE_PARTITIONS)
print("  TUNING_PARALLELISM       =", settings.TUNING_PARALLELISM)
print()
print("Actual Spark session configuration:")
for key, value in actual.items():
    print(f"  {key} = {value}")
print()

if actual["spark.driver.memory"] != settings.SPARK_DRIVER_MEMORY:
    print("*** MISMATCH: driver memory did NOT apply. ***")
    print("This means a Spark session was already running before this cell ran, and ")
    print("stopping it here wasn't enough to free up a truly fresh JVM. Go to ")
    print("Kernel > Restart, then re-run this notebook from the top cell before ")
    print("continuing to the training cell below.")
else:
    print("Spark session configured correctly - safe to proceed to training.")

## 4. Run the full training pipeline

Only proceed here once the cell above confirmed the configuration matches. This tunes all 4 required models (wider grids, `TUNING_PARALLELISM` concurrent fits per model), selects the winner by the RMSE→MAE→R² rule, evaluates it exactly once on the untouched test set, and saves all output artifacts to `models/`.

Expect a lot of Spark log noise; the useful progress lines come from `src.training.tune_models` and look like:
```
Tuning RandomForestRegressor over 6 parameter combinations x 3 folds (parallelism=4)
RandomForestRegressor: validation RMSE=... MAE=... R2=... (...s, best params=...)
```

If this cell fails with `Py4JJavaError: ... ConnectionRefusedError`, set `TUNING_PARALLELISM=1` in `.env`, restart the kernel, and re-run from the top — that's the documented, known-safe fallback (`PLAN.md` §23 #13/#18).

In [ ]:
from src.training.evaluate_model import run_training_pipeline

run_training_pipeline()

## 5. Results

In [ ]:
import json

print("--- models/model_comparison.csv ---")
print(open("models/model_comparison.csv").read())

print("--- models/model_metadata.json ---")
print(json.dumps(json.load(open("models/model_metadata.json")), indent=2))

print("--- models/model_metrics.json ---")
print(json.dumps(json.load(open("models/model_metrics.json")), indent=2))